In [25]:
import numpy as np
import matplotlib.pyplot as plt
import corner
from scipy.stats import ranksums, levene

In [26]:
# Define base paths and model parameters
base_path = "/Users/arjansuri/Desktop/Coding/TCU/BirdFlu/code/model3"
strains = ['h3n2(2003)', 'h5n1(2005)', 'h5n1(2022)']
tissues = ['nasal', 'tracheal']

# LaTeX formatted parameter names for your model
param_names = [
    r'$\beta$',   # infection rate
    r'$\gamma$',  # rate of conversion to resistant state
    r'$\rho$',    # rate of return from resistant state
    r'$\delta$',  # death rate of infected cells
    r'$p$',       # virus production rate
    r'$c$',       # virus clearance rate
    r'$\alpha$'   # interferon clearance rate
]

def calculate_confidence_intervals(params):
    lower_bound = np.percentile(params, 2.5, axis=0)
    upper_bound = np.percentile(params, 97.5, axis=0)
    return lower_bound, upper_bound

# Create figures directory if it doesn't exist
import os
os.makedirs(f"{base_path}/figures", exist_ok=True)

# Load data for each tissue and strain combination
data = {}
for tissue in tissues:
    data[tissue] = {}
    for strain in strains:
        # Updated paths to match actual directory structure
        params_file = f"{base_path}/{strain}/{tissue}/bootstrap_results/{strain}_{tissue}_params.txt"
        ssr_file = f"{base_path}/{strain}/{tissue}/bootstrap_results/{strain}_{tissue}_ssr.txt"
        
        try:
            params = np.loadtxt(params_file, delimiter=',')
            ssr = np.loadtxt(ssr_file, delimiter=',')
            
            data[tissue][strain] = {
                'params': params,
                'ssr': ssr,
                'log_params': params  # Parameters are already in log space - don't double log
            }
            print(f"Loaded {tissue} {strain}: {len(params)} bootstrap samples")
                
        except Exception as e:
            print(f"Error loading data for {tissue} {strain}: {e}")
            print(f"  Tried to load: {params_file}")
            print(f"  Tried to load: {ssr_file}")

Loaded nasal h3n2(2003): 1000 bootstrap samples
Loaded nasal h5n1(2005): 1000 bootstrap samples
Loaded nasal h5n1(2022): 1000 bootstrap samples
Loaded tracheal h3n2(2003): 1000 bootstrap samples
Loaded tracheal h5n1(2005): 1000 bootstrap samples
Loaded tracheal h5n1(2022): 1000 bootstrap samples


In [27]:
# Generate parameter distribution plots
for i, param in enumerate(param_names):
    plt.figure(figsize=(12, 8))
    
    for tissue in tissues:
        for strain in strains:
            if tissue in data and strain in data[tissue]:
                plt.hist(data[tissue][strain]['log_params'][:, i], 
                        bins=50, alpha=0.3, 
                        label=f'{tissue} {strain}')
    
    plt.text(0.05, 0.95, param, fontsize=60, 
             transform=plt.gca().transAxes, 
             verticalalignment='top', 
             horizontalalignment='left')
    
    plt.xlabel(f'log({param})', fontsize=18)
    plt.ylabel('Frequency', fontsize=18)
    plt.legend(fontsize=12)
    plt.savefig(f"{base_path}/figures/log_{param}_histogram.png", bbox_inches='tight')
    plt.close()

# Generate SSR histograms
plt.figure(figsize=(12, 8))
for tissue in tissues:
    for strain in strains:
        if tissue in data and strain in data[tissue]:
            plt.hist(data[tissue][strain]['ssr'], 
                    bins=50, alpha=0.3, 
                    label=f'{tissue} {strain}')

plt.xlabel('SSR', fontsize=24)
plt.ylabel('Frequency', fontsize=24)
plt.legend(fontsize=14)
plt.savefig(f"{base_path}/figures/ssr_histogram.png", bbox_inches='tight')
plt.close()

# Calculate and print confidence intervals
for tissue in tissues:
    for strain in strains:
        if tissue in data and strain in data[tissue]:
            ci = calculate_confidence_intervals(data[tissue][strain]['log_params'])
            print(f"\n{tissue} {strain} confidence intervals:")
            for i, param in enumerate(param_names):
                print(f"log({param}): 95% CI = [{ci[0][i]:.7f}, {ci[1][i]:.7f}]")

# Generate corner plots without removing any samples - keep all data
for tissue in tissues:
    for strain in strains:
        if tissue in data and strain in data[tissue]:
            log_params = data[tissue][strain]['log_params']
            
            print(f"Using all {len(log_params)} samples for {tissue} {strain}")
            
            try:
                corner_fig = corner.corner(
                    log_params,
                    labels=param_names,
                    show_titles=True,
                    title_fmt=".2f",
                    quantiles=[0.025, 0.5, 0.975],
                    hist_kwargs={"density": True, "alpha": 0.5},
                    label_kwargs={"fontsize": 12}
                )
                plt.savefig(f"{base_path}/figures/{tissue}_{strain}_corner_plot.png", 
                           bbox_inches='tight')
                plt.close()
                print(f"Corner plot saved for {tissue} {strain}")
            except Exception as e:
                print(f"Corner plot failed for {tissue} {strain}: {e}")

# Statistical tests
iterations = 100
sample_size = 10

# Perform statistical tests between all pairs
for tissue in tissues:
    print(f"\nStatistical tests for {tissue} tissue:")
    for i, param in enumerate(param_names):
        for strain1 in strains:
            for strain2 in strains:
                if strain1 < strain2:  # Avoid duplicate comparisons
                    if (tissue in data and strain1 in data[tissue] 
                        and strain2 in data[tissue]):
                        wilcox_pvals = []
                        levene_pvals = []
                        
                        for _ in range(iterations):
                            sample1 = np.random.choice(
                                data[tissue][strain1]['log_params'][:, i], 
                                sample_size, replace=False
                            )
                            sample2 = np.random.choice(
                                data[tissue][strain2]['log_params'][:, i], 
                                sample_size, replace=False
                            )
                            
                            _, wilcox_p = ranksums(sample1, sample2)
                            _, levene_p = levene(sample1, sample2, center="median")
                            
                            wilcox_pvals.append(wilcox_p)
                            levene_pvals.append(levene_p)
                        
                        print(f"\n{param} comparison {strain1} vs {strain2}:")
                        print(f"Wilcoxon p-value: {np.mean(wilcox_pvals):.4f}")
                        print(f"Levene p-value: {np.mean(levene_pvals):.4f}")

# Calculate means and standard deviations for each parameter
print("\nParameter statistics:")
for tissue in tissues:
    for strain in strains:
        if tissue in data and strain in data[tissue]:
            print(f"\n{tissue} {strain}:")
            # Convert back from log space for interpretable statistics
            original_params = np.exp(data[tissue][strain]['log_params'])
            for i, param in enumerate(param_names):
                mean = np.mean(original_params[:, i])
                std = np.std(original_params[:, i])
                print(f"{param}: mean = {mean:.4e} ± {std:.4e}")


nasal h3n2(2003) confidence intervals:
log($\beta$): 95% CI = [0.1792906, 10.0000000]
log($\gamma$): 95% CI = [0.0000010, 0.0000131]
log($\rho$): 95% CI = [0.0000010, 10.0000000]
log($\delta$): 95% CI = [0.0000010, 0.0429750]
log($p$): 95% CI = [0.1167617, 0.2822664]
log($c$): 95% CI = [0.0000010, 0.0281393]
log($\alpha$): 95% CI = [0.0043793, 0.0454211]

nasal h5n1(2005) confidence intervals:
log($\beta$): 95% CI = [0.1506370, 9.9999233]
log($\gamma$): 95% CI = [0.0000010, 3.1359302]
log($\rho$): 95% CI = [0.0000010, 10.0000000]
log($\delta$): 95% CI = [0.0040041, 0.0448163]
log($p$): 95% CI = [0.1225886, 0.6586858]
log($c$): 95% CI = [0.0063611, 0.1453283]
log($\alpha$): 95% CI = [0.0125565, 0.0666317]

nasal h5n1(2022) confidence intervals:
log($\beta$): 95% CI = [0.2323647, 10.0000000]
log($\gamma$): 95% CI = [0.0000010, 0.0000607]
log($\rho$): 95% CI = [0.0000010, 10.0000000]
log($\delta$): 95% CI = [0.0049582, 0.0619382]
log($p$): 95% CI = [0.1000000, 0.2116402]
log($c$): 95% CI

Corner plot saved for nasal h3n2(2003)
Using all 1000 samples for nasal h5n1(2005)


Corner plot saved for nasal h5n1(2005)
Using all 1000 samples for nasal h5n1(2022)


Corner plot saved for nasal h5n1(2022)
Using all 1000 samples for tracheal h3n2(2003)


Corner plot saved for tracheal h3n2(2003)
Using all 1000 samples for tracheal h5n1(2005)


Corner plot saved for tracheal h5n1(2005)
Using all 1000 samples for tracheal h5n1(2022)


Corner plot saved for tracheal h5n1(2022)

Statistical tests for nasal tissue:

$\beta$ comparison h3n2(2003) vs h5n1(2005):
Wilcoxon p-value: 0.1776
Levene p-value: 0.3140

$\beta$ comparison h3n2(2003) vs h5n1(2022):
Wilcoxon p-value: 0.4562
Levene p-value: 0.4839

$\beta$ comparison h5n1(2005) vs h5n1(2022):
Wilcoxon p-value: 0.0686
Levene p-value: 0.4136

$\gamma$ comparison h3n2(2003) vs h5n1(2005):
Wilcoxon p-value: 0.2625
Levene p-value: 0.1460

$\gamma$ comparison h3n2(2003) vs h5n1(2022):
Wilcoxon p-value: 0.4315
Levene p-value: 0.3674

$\gamma$ comparison h5n1(2005) vs h5n1(2022):
Wilcoxon p-value: 0.1701
Levene p-value: 0.1496

$\rho$ comparison h3n2(2003) vs h5n1(2005):
Wilcoxon p-value: 0.4902
Levene p-value: 0.5416

$\rho$ comparison h3n2(2003) vs h5n1(2022):
Wilcoxon p-value: 0.4267
Levene p-value: 0.4343

$\rho$ comparison h5n1(2005) vs h5n1(2022):
Wilcoxon p-value: 0.3235
Levene p-value: 0.4468

$\delta$ comparison h3n2(2003) vs h5n1(2005):
Wilcoxon p-value: 0.3801
Lev

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/_core/_methods.py:194: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
